# A full business solution
## Business Challange:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits . 
We will be provided with company name and their primary website.

In [5]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from scraper import WebsiteScraper


In [6]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key) >10:
    print("API key is valid")
else:
    print("API key is invalid")
MODEL = "gpt-5-nano"
openai = OpenAI()

API key is valid


In [7]:
scraper = WebsiteScraper('https://www.flightaware.com/live/')
links = scraper.get_links()
links

['/',
 '/commercial/aeroapi/',
 '/commercial/firehose/',
 '/commercial/foresight/',
 '/commercial/reports/rapid/',
 '/commercial/reports/custom/',
 '/commercial/integrated-maps/',
 '/commercial/aviator/',
 '/commercial/premium/',
 '/commercial/global/',
 '/commercial/fbo-toolbox/',
 '/commercial/tv/',
 '/commercial/globalbeacon/',
 'https://www.flightaware.com/industry/airports',
 'https://www.flightaware.com/industry/airlines',
 'https://www.flightaware.com/industry/business',
 'https://www.flightaware.com/industry/government',
 'https://www.flightaware.com/industry/manufacturer',
 'https://www.flightaware.com/industry/travel',
 '/adsb/',
 '/adsb/stats/',
 'https://go.flightaware.com/skyawareanywhere',
 '/adsb/coverage/',
 'https://flightaware.store/',
 '/adsb/piaware/build/',
 '/adsb/flightfeeder/',
 '/adsb/faq/',
 '/live',
 '/live/cancelled/',
 '/live/airport/delays/',
 '/miserymap/',
 '/live/findflight/',
 '/live/fleet/',
 '/resources/airport/browse/',
 '/live/aircrafttype/',
 '/li

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be the most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links":[
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of the links are most relevant to include in a brochure about the company,
    respond with full https URL in JSON format.
    Do not include Terms of Service, privacy policy, email links.

    Links (some might be relative links):

    """
    scraper = WebsiteScraper(url)
    links = scraper.get_links()
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://www.flightaware.com/live/"))


    Here is the list of links on the website https://www.flightaware.com/live/ -
    Please decide which of the links are most relevant to include in a brochure about the company,
    respond with full https URL in JSON format.
    Do not include Terms of Service, privacy policy, email links.

    Links (some might be relative links):

    /
/commercial/aeroapi/
/commercial/firehose/
/commercial/foresight/
/commercial/reports/rapid/
/commercial/reports/custom/
/commercial/integrated-maps/
/commercial/aviator/
/commercial/premium/
/commercial/global/
/commercial/fbo-toolbox/
/commercial/tv/
/commercial/globalbeacon/
https://www.flightaware.com/industry/airports
https://www.flightaware.com/industry/airlines
https://www.flightaware.com/industry/business
https://www.flightaware.com/industry/government
https://www.flightaware.com/industry/manufacturer
https://www.flightaware.com/industry/travel
/adsb/
/adsb/stats/
https://go.flightaware.com/skyawareanywhere
/adsb/coverage/
https://flightaw

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)  }
        ],
        response_format = {"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://www.flightaware.com/live/")

Selecting relevant links for https://www.flightaware.com/live/ by calling gpt-5-nano
Found 14 relevant links


{'links': [{'type': 'about page', 'url': 'https://www.flightaware.com/about'},
  {'type': 'careers page',
   'url': 'https://www.flightaware.com/about/careers/'},
  {'type': 'history page',
   'url': 'https://www.flightaware.com/about/history.rvt'},
  {'type': 'datasources page',
   'url': 'https://www.flightaware.com/about/datasources/'},
  {'type': 'contact page',
   'url': 'https://www.flightaware.com/about/contact/'},
  {'type': 'news page', 'url': 'https://www.flightaware.com/news/'},
  {'type': 'blog', 'url': 'https://blog.flightaware.com'},
  {'type': 'webinars page', 'url': 'https://go.flightaware.com/webinars'},
  {'type': 'industry page',
   'url': 'https://www.flightaware.com/industry/airports'},
  {'type': 'industry page',
   'url': 'https://www.flightaware.com/industry/airlines'},
  {'type': 'industry page',
   'url': 'https://www.flightaware.com/industry/business'},
  {'type': 'industry page',
   'url': 'https://www.flightaware.com/industry/government'},
  {'type': 'indus

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def get_page_content(url):
    return WebsiteScraper(url).get_content()

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = get_page_content(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links: \n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += get_page_content(url)
    return result


In [15]:
print(fetch_page_and_all_relevant_links("https://www.flightaware.com/live/"))

Selecting relevant links for https://www.flightaware.com/live/ by calling gpt-5-nano
Found 32 relevant links
## Landing Page:

Live Flight Tracker - FlightAware

Products
Data Products
AeroAPI
Flight data API with on-demand flight status and flight tracking data.
FlightAware Firehose
Streaming flight data feed for enterprise integrations with real-time, historical and predictive flight data.
FlightAware Foresight
Predictive technology to strengthen customer trust in your operations
Rapid Reports
Quickly purchase historical reports delivered via email.
Custom Reports
Consultative detailed and customized flight tracking data reports.
Integrated Mapping Solutions
Incorporate FlightAware maps in your web and mobile applications
Applications
FlightAware Aviator
The ultimate flight tracking suite for small aircraft/general aviation (GA) owners and operators.
Premium Subscriptions
A personalized flight-following experience with unlimited alerts and more.
FlightAware Global
The industry standa

In [16]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("FlightAware", "https://www.flightaware.com/live/")

Selecting relevant links for https://www.flightaware.com/live/ by calling gpt-5-nano
Found 21 relevant links


'\nYou are looking at a company called: FlightAware\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nLive Flight Tracker - FlightAware\n\nProducts\nData Products\nAeroAPI\nFlight data API with on-demand flight status and flight tracking data.\nFlightAware Firehose\nStreaming flight data feed for enterprise integrations with real-time, historical and predictive flight data.\nFlightAware Foresight\nPredictive technology to strengthen customer trust in your operations\nRapid Reports\nQuickly purchase historical reports delivered via email.\nCustom Reports\nConsultative detailed and customized flight tracking data reports.\nIntegrated Mapping Solutions\nIncorporate FlightAware maps in your web and mobile applications\nApplications\nFlightAware Aviator\nThe ultimate flight tracking suite for small aircraft/general aviation (GA) owners and operators.\nP

In [18]:
from IPython.display import Markdown


def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("FlightAware", "https://www.flightaware.com/live/")

Selecting relevant links for https://www.flightaware.com/live/ by calling gpt-5-nano
Found 43 relevant links


# FlightAware Brochure

---

## About FlightAware

FlightAware is a leading innovator in real-time global flight tracking and aviation data services. With an extensive suite of products and applications, FlightAware empowers aviation professionals, businesses, governments, and enthusiasts worldwide to access live and predictive flight information through cutting-edge technology and comprehensive data services.

---

## Products & Services

### Data Products
- **AeroAPI**: On-demand flight status and tracking via an intuitive API.
- **FlightAware Firehose**: Enterprise-grade streaming flight data feed including real-time, historical, and predictive data.
- **FlightAware Foresight**: Advanced predictive analytics to improve operational reliability and customer trust.
- **Rapid Reports**: Fast email delivery of historical flight data reports.
- **Custom Reports**: Tailored consultative reports to meet unique aviation data needs.
- **Integrated Mapping Solutions**: FlightAware maps embeddable in web and mobile applications for enhanced visualization.

### Applications
- **FlightAware Aviator**: Comprehensive flight tracking suite designed for small aircraft and general aviation owners/operators.
- **Premium Subscriptions**: Personalized flight tracking with unlimited alerts and premium features.
- **FlightAware Global**: The industry standard flight tracking platform tailored for business aviation owners and operators.
- **FlightAware FBO Toolbox**: Tools for Fixed Base Operators (FBOs) to optimize operations and grow sales.
- **FlightAware TV**: Full-screen interactive maps ideal for operators and FBOs.
- **GlobalBeacon**: GADSS-compliant global tracking and alerting solution for airlines and aircraft operators.

---

## Industries Served

FlightAware serves a broad range of sectors, including but not limited to:
- Airports
- Airlines
- Business Aviation
- Government Agencies
- Manufacturers
- Travel Industry

FlightAware provides the critical data and services required for safe, efficient, and informed aviation operations worldwide.

---

## Company Culture

FlightAware fosters an innovative and collaborative culture driven by technology and a passion for aviation. The company values data accuracy, reliability, and creating seamless user experiences. Its engineering-first mindset promotes continuous improvement and the deployment of predictive technologies that anticipate customer needs and operational challenges.

---

## Careers at FlightAware

FlightAware offers exciting career opportunities for professionals passionate about aviation, data, and technology. Employees benefit from a dynamic atmosphere where creativity thrives and teamwork is paramount. Available roles span software development, data science, product management, customer support, and more.

For those eager to impact the future of flight tracking and aviation data intelligence, FlightAware provides the platform and culture to grow and innovate.

---

## Why Choose FlightAware?

- **Global Reach**: Unmatched global coverage with live, historical, and predictive flight data.
- **Robust APIs and Data Feeds**: Easy to integrate data sources empower your applications and business systems.
- **Innovative Technology**: Leveraging predictive analytics to improve operational decisions.
- **Trusted by Industry Leaders**: Airlines, airports, government agencies, and business aviation choose FlightAware.
- **Community Engagement**: Active user community sharing real-time data, photos, and discussions.

---

## Contact & Access

- Create a FlightAware account or log in to access personalized services.
- Download the FlightAware App for flight tracking on the go.
- Visit FlightAware online for more information, support, and resources.

---

FlightAware — Empowering the world to track and understand every flight with confidence and precision.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI, **but this time in German**,
with the familiar typewriter animation

In [20]:
from IPython.display import update_display

brochure_system_prompt_german = """
You are a marketing assistant.

Generate the brochure entirely in German.
Use natural, professional German.
Keep headings, bullet points, and formatting in German.
"""

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt_german},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("FlightAware", "https://www.flightaware.com/live/")

Selecting relevant links for https://www.flightaware.com/live/ by calling gpt-5-nano
Found 12 relevant links


# FlightAware – Ihr weltweiter Live-Flugtracker

---

## Über FlightAware

FlightAware ist die führende Plattform für Live-Flugverfolgung und Flugstatus-Informationen. Unsere innovativen Lösungen richten sich an private Piloten, Geschäftsreisende, Fluggesellschaften, Flughäfen sowie Unternehmen und Behörden weltweit.

Mit FlightAware erhalten Sie jederzeit Echtzeitdaten, historische Daten und prädiktive Analysen für Flugbewegungen auf der ganzen Welt. Unsere Technologie stärkt das Vertrauen in Ihre Abläufe und optimiert die Effizienz Ihrer Luftfahrtoperationen.

---

## Unsere Produkte

### Datenprodukte
- **AeroAPI**  
  Flight-Daten-API mit On-Demand Flugstatus und Tracking-Daten.
- **FlightAware Firehose**  
  Streaming-Feed für Echtzeit-, historische und prognostizierte Flugdaten – ideal für Unternehmensintegration.
- **FlightAware Foresight**  
  Predictive-Technologie zur Verbesserung der Zuverlässigkeit Ihrer Flugoperationen.
- **Rapid Reports**  
  Schneller Zugriff auf historische Berichte per E-Mail.
- **Custom Reports**  
  Maßgeschneiderte, detaillierte Flugtracking-Datenanalysen.
- **Integrierte Kartenlösungen**  
  Einbindung von FlightAware Karten in Web- und Mobilanwendungen.

### Anwendungen
- **FlightAware Aviator**  
  Umfassende Flugverfolgung speziell für Kleinflugzeuge und die Allgemeine Luftfahrt.
- **Premium-Abonnements**  
  Personalisierte Flugverfolgung mit unbegrenzten Benachrichtigungen und erweiterten Funktionen.
- **FlightAware Global**  
  Branchenstandard für die Flugverfolgung in der Business Aviation.
- **FlightAware FBO Toolbox**  
  Optimieren Sie den Betrieb Ihres Fixed Base Operators und steigern Sie Ihren Umsatz.
- **FlightAware TV**  
  Vollbildanzeige von FlightAware-Karten für Betreiber und FBOs.
- **GlobalBeacon**  
  GADSS-konformes globales Tracking und Alert-System für Airlines und Flugzeugbetreiber.

---

## Branchen und Einsatzgebiete

- Flughäfen
- Fluggesellschaften
- Geschäftsreisebranche
- Regierungsorganisationen
- Flugzeughersteller
- Reise- und Tourismusbranche

---

## Weitere Services und Angebote

- **ADS-B Netzwerk & Technologie**  
  Erweiterte Abdeckung durch SkyAware Anywhere, PiAware Receiver und FlightFeeder.
- **Statistiken und Analysen**  
  Aktuelle Einblicke in Verspätungen, Flugstörungen und Routen.
- **Community & Support**  
  User-Fotos, Diskussionen, aktuelle Meldungen und umfangreiche FAQ.

---

## FlightAware – Ihre Vorteile auf einen Blick

- Echtzeit-Flugverfolgung weltweit
- Breite Produktpalette für Profis und Privatnutzer
- Integration in eigene Systeme und Apps dank leistungsstarker APIs
- Detaillierte Berichte und prädiktive Tools zur Planungssicherheit
- Zuverlässige Partnerschaft mit führenden Branchenakteuren

---

## Kontakt & Einstieg

Erstellen Sie noch heute ein FlightAware-Konto oder laden Sie unsere App herunter, um direkt live dabei zu sein!

Besuchen Sie uns online:  
[www.flightaware.com](https://www.flightaware.com)

---

**FlightAware** – Das neue Zeitalter der Flugverfolgung beginnt hier.